# Check Five Rows

Compare five feature rows with the source readings.

Open this notebook in `notebooks/M3/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

## Choose the two inputs

In [ ]:
from pathlib import Path
import pandas as pd
raw_path = PROJECT_ROOT / "data/pilot/half_hour_sample.csv"
feature_path = PROJECT_ROOT / "data/processed/prediction_inputs_sample.csv"
raw_path, feature_path

## Read the half-hour data

In [ ]:
raw = pd.read_csv(raw_path, na_values=["Null", "null"])
raw["tstp"] = pd.to_datetime(raw["tstp"], utc=True)
raw["energy(kWh/hh)"] = pd.to_numeric(raw["energy(kWh/hh)"])
raw.head()

## Read the feature file

In [ ]:
features = pd.read_csv(feature_path)
features["target_hour"] = pd.to_datetime(features["target_hour"], utc=True)
features = features.sort_values(["LCLid", "target_hour"])
features.head()

## See whether the keys are unique

In [ ]:
raw.duplicated(["LCLid", "tstp"]).sum(), features.duplicated(["LCLid", "target_hour"]).sum()

## Choose the five kept examples

In [ ]:
five_cases = [
    (1, "normal afternoon", "MAC000414", "2013-06-12 14:00"),
    (2, "usable after a gap-related exclusion", "MAC005456", "2013-01-26 10:00"),
    (3, "first possible kept hour", "MAC001312", "2013-01-02 00:00"),
    (4, "measured zero previous hour", "MAC000675", "2013-02-18 12:00"),
    (5, "midnight on clock-change date, UTC", "MAC000414", "2013-03-31 00:00")]
five_cases

## Choose the four excluded examples

In [ ]:
excluded_cases = [
    (6, "missing target", "MAC005456", "2013-08-18 05:00"),
    (7, "missing previous hour", "MAC005456", "2013-01-26 09:00"),
    (8, "missing yesterday", "MAC005456", "2013-01-27 08:00"),
    (9, "no earlier history", "MAC001312", "2013-01-01 00:00")]
excluded_cases

## Put the examples into one table

In [ ]:
cases = pd.DataFrame(five_cases + excluded_cases, columns=["case_id", "case", "LCLid", "target_hour"])
cases["target_hour"] = pd.to_datetime(cases["target_hour"], utc=True)
cases["expected_kept"] = cases["case_id"] <= 5
cases = cases.set_index("case_id")
cases

## List the three hours needed per case

In [ ]:
offsets = pd.DataFrame({"field": ["target_kwh", "lag_1h_kwh", "lag_24h_kwh"], "offset": [0, 1, 24]})
parts = cases.reset_index().merge(offsets, how="cross")
parts["hour_start"] = parts["target_hour"] - pd.to_timedelta(parts["offset"], unit="h")
parts[["case_id", "field", "hour_start"]].head(6)

## Find the correct half-hour end labels

In [ ]:
parts["first_end"] = parts["hour_start"] + pd.Timedelta(minutes=30)
parts["second_end"] = parts["hour_start"] + pd.Timedelta(hours=1)
parts[["case_id", "field", "hour_start", "first_end", "second_end"]].head(3)

## Index the raw readings

In [ ]:
raw_values = raw.set_index(["LCLid", "tstp"])["energy(kWh/hh)"]
raw_values.head()

## Look up the first half-hour

In [ ]:
first_keys = pd.MultiIndex.from_arrays([parts["LCLid"], parts["first_end"]])
parts["first_kwh"] = raw_values.reindex(first_keys).to_numpy()
parts[["case_id", "field", "first_kwh"]].head(3)

## Look up the second half-hour

In [ ]:
second_keys = pd.MultiIndex.from_arrays([parts["LCLid"], parts["second_end"]])
parts["second_kwh"] = raw_values.reindex(second_keys).to_numpy()
parts[["case_id", "field", "second_kwh"]].head(3)

## Independently add the two readings

In [ ]:
parts["recomputed_kwh"] = parts["first_kwh"] + parts["second_kwh"]
parts[["case_id", "field", "hour_start", "first_end",
    "second_end", "first_kwh", "second_kwh", "recomputed_kwh"]]

## Arrange the independently calculated values

In [ ]:
energy_fields = ["target_kwh", "lag_1h_kwh", "lag_24h_kwh"]
recomputed = parts.pivot(index="case_id", columns="field", values="recomputed_kwh")
recomputed = recomputed[energy_fields]
recomputed

## Look up the submitted feature values

In [ ]:
feature_values = features.set_index(["LCLid", "target_hour"])
case_keys = pd.MultiIndex.from_arrays([cases["LCLid"], cases["target_hour"]])
actual = feature_values.reindex(case_keys)
actual.index = cases.index
actual

## Check which examples are present

In [ ]:
present = pd.Series(case_keys.isin(feature_values.index), index=cases.index, name="present_in_features")
membership = cases[["case", "expected_kept"]].join(present)
membership["matches_expectation"] = membership["expected_kept"] == membership["present_in_features"]
membership

## Compare the three energy values

In [ ]:
energy_matches = (actual[energy_fields] - recomputed).abs() < 0.0005
energy_matches.loc[1:5]

## Calculate calendar expectations

In [ ]:
calendar = pd.DataFrame(index=cases.index)
calendar["hour"] = cases["target_hour"].dt.hour
calendar["weekday"] = cases["target_hour"].dt.dayofweek
calendar["weekday_name"] = cases["target_hour"].dt.day_name()
calendar

## Compare the two calendar fields

In [ ]:
calendar_matches = actual[["hour", "weekday"]] == calendar[["hour", "weekday"]]
calendar_matches.loc[1:5]

## Collect the five real-row results

In [ ]:
checked = cases.loc[1:5].join(actual.loc[1:5])
checked = checked.join(recomputed.loc[1:5].add_prefix("recomputed_"))
checked = checked.join(calendar[["weekday_name"]])
checked["all_match"] = present.loc[1:5] & energy_matches.loc[1:5].all(axis=1) & calendar_matches.loc[1:5].all(axis=1)
checked

## Keep the raw calculation evidence

In [ ]:
raw_evidence = parts.pivot(index="case_id",
    columns="field", values=["first_kwh", "second_kwh",
    "hour_start", "first_end", "second_end"])
raw_evidence.columns = [field + "_" + kind for kind, field in raw_evidence.columns]
checked = checked.join(raw_evidence)
checked.head(1)

## What a plain row shift would use

In [ ]:
shifted = features[["LCLid", "target_hour"]].copy()
by_home = features.groupby("LCLid")
shifted["previous_row_time"] = by_home["target_hour"].shift(1)
shifted["row_24_back_time"] = by_home["target_hour"].shift(24)
shifted["if_row_shift_lag_1h"] = by_home["target_kwh"].shift(1)
shifted["if_row_shift_lag_24h"] = by_home["target_kwh"].shift(24)
shifted.head()

## Compare row shifts with exact times

In [ ]:
shift_examples = shifted.set_index(["LCLid", "target_hour"]).reindex(case_keys[:5])
shift_examples.index = cases.loc[1:5].index
shift_examples["required_lag_1h_time"] = cases.loc[1:5, "target_hour"] - pd.Timedelta(hours=1)
shift_examples["required_lag_24h_time"] = cases.loc[1:5, "target_hour"] - pd.Timedelta(hours=24)
checked = checked.join(shift_examples)
shift_examples

## Find raw-data reasons for exclusion

In [ ]:
missing = recomputed.isna()
missing.loc[6:9]

## Assign the same priority as the builder

In [ ]:
reason = pd.Series("kept", index=cases.index)
reason[missing["lag_24h_kwh"]] = "missing_lag_24h"
reason[missing["lag_1h_kwh"]] = "missing_lag_1h"
reason[missing["target_kwh"]] = "missing_target"
reason.loc[6:9]

## Check the specific expected reasons

In [ ]:
expected_reason = pd.Series(["missing_target",
    "missing_lag_1h", "missing_lag_24h", "missing_lag_1h"],
    index=cases.loc[6:9].index)
excluded = cases.loc[6:9].join(recomputed.add_prefix("recomputed_"))
excluded = excluded.join(missing.add_prefix("missing_")).join(present)
excluded["reason"] = reason
excluded["expected_reason"] = expected_reason
excluded["as_expected"] = ~present.loc[6:9] & missing.loc[6:9].any(axis=1) & (reason.loc[6:9] == expected_reason)
excluded

## Attach evidence to excluded examples

In [ ]:
excluded = excluded.join(raw_evidence)
excluded[["case", "reason", "present_in_features", "as_expected"]]

## Summary

In [ ]:
checked["all_match"].all(), excluded["as_expected"].all()

## Choose the output folder

In [ ]:
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

## Save the five checked rows

In [ ]:
five_path = output_dir / "five_row_check.csv"
checked.to_csv(five_path, index_label="case_id")
five_path

## Save the excluded examples

In [ ]:
excluded_path = output_dir / "missing_input_examples.csv"
excluded.to_csv(excluded_path, index_label="case_id")
excluded_path